In [ ]:
!pip install -q ultralytics opencv-python lxml pyyaml tqdm openvino kaggle

In [7]:
# 13) Tạo video từ một sequence ảnh UA-DETRAC

import cv2
from pathlib import Path
import os
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import time

DATA_ROOT = Path("data")
# YOLO_ROOT = Path("/CITD/ua_detrac_yolo")

# Lấy một folder ảnh bất kỳ trong dataset gốc
seq_folder = None
for folder in DATA_ROOT.rglob("*"):
    if folder.is_dir():
        imgs = sorted(list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
        if len(imgs) > 50:
            seq_folder = folder
            break

print("Sequence folder:", seq_folder)

seq_imgs = sorted(list(seq_folder.glob("*.jpg")) + list(seq_folder.glob("*.png")))

first_img = cv2.imread(str(seq_imgs[0]))
h, w = first_img.shape[:2]

video_path = "ua_detrac_sequence_test.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(video_path, fourcc, 25, (w, h))

for img_path in seq_imgs:
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    writer.write(frame)

writer.release()

print("Saved video:", video_path)

Sequence folder: data/DETRAC-Images/DETRAC-Images/MVI_40192
Saved video: ua_detrac_sequence_test.mp4


In [9]:
from ultralytics import YOLO
import torch

# Kiểm tra MPS
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple Silicon GPU (MPS)")
else:
    device = "cpu"
    print("Using CPU")

model = YOLO("yolo11n.pt")

tracking_results = model.track(
    source="ua_detrac_sequence_test.mp4",
    tracker="bytetrack.yaml",
    imgsz=640,
    conf=0.25,
    iou=0.5,
    device=device,
    save=True,
    save_txt=True,
    project="runs",
    name="yolo11n_bytetrack_ua_detrac",
    persist=True
)

print("Done")

Using Apple Silicon GPU (MPS)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/2195) /Users/vivianele/Desktop/CITD_thesis/ua_detrac_sequence_test.mp4: 384x640 22 cars, 2 trucks, 281.2ms
video 1/1 (frame 2/2195) /Users/vivianele/Desktop/CITD_thesis/ua_detrac_sequence_test.mp4: 384x640 22 cars, 2 trucks, 18.2ms
video 1/1 (frame 3/2195) /Users/vivianele/Desktop/CITD_thesis/ua_detrac_sequence_test.mp4: 384x640 21 cars, 2 trucks, 19.3ms
video 1/1 (frame 4/2195) /Users/vivianele/Desktop/CI

# VẼ LAINE

In [ ]:
import cv2
import json
from pathlib import Path

video_path = "ua_detrac_sequence_test.mp4"   # sửa path video
lane_json = "data/lanes.json"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if not ret:
    raise ValueError("Không đọc được video")

points = []
lanes = []
current_lane = []

def mouse_callback(event, x, y, flags, param):
    global current_lane, lanes

    if event == cv2.EVENT_LBUTTONDOWN:
        current_lane.append([x, y])
        print("Point:", x, y)

    elif event == cv2.EVENT_RBUTTONDOWN:
        if len(current_lane) >= 3:
            lane_name = f"lane_{len(lanes)+1}"
            lanes.append({
                "name": lane_name,
                "points": current_lane.copy()
            })
            print(f"Saved {lane_name}:", current_lane)
            current_lane = []
        else:
            print("Cần ít nhất 3 điểm để tạo 1 lane")

clone = frame.copy()
cv2.namedWindow("Draw lanes")
cv2.setMouseCallback("Draw lanes", mouse_callback)

print("""
Hướng dẫn:
- Click trái: thêm điểm polygon cho lane
- Click phải: kết thúc lane hiện tại
- Nhấn S: lưu lanes.json
- Nhấn Q: thoát
""")

while True:
    display = clone.copy()

    # vẽ lane đã lưu
    for lane in lanes:
        pts = lane["points"]
        for i in range(len(pts)):
            cv2.line(display, tuple(pts[i]), tuple(pts[(i+1) % len(pts)]), (0,255,0), 2)
        cv2.putText(display, lane["name"], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    # vẽ lane đang chọn
    for p in current_lane:
        cv2.circle(display, tuple(p), 5, (0,0,255), -1)
    for i in range(len(current_lane)-1):
        cv2.line(display, tuple(current_lane[i]), tuple(current_lane[i+1]), (0,0,255), 2)

    cv2.imshow("Draw lanes", display)
    key = cv2.waitKey(1) & 0xFF

    if key == ord("s"):
        with open(lane_json, "w") as f:
            json.dump(lanes, f, indent=4)
        print("Saved:", lane_json)

    elif key == ord("q"):
        break

cv2.destroyAllWindows()


Hướng dẫn:
- Click trái: thêm điểm polygon cho lane
- Click phải: kết thúc lane hiện tại
- Nhấn S: lưu lanes.json
- Nhấn Q: thoát

Point: 4 288
Point: 11 361
Point: 488 29
Point: 469 28
Point: 5 290
Saved lane_1: [[4, 288], [11, 361], [488, 29], [469, 28], [5, 290]]
Point: 11 365
Point: 13 503
Point: 501 38
Point: 489 32
Point: 11 367
Saved lane_2: [[11, 365], [13, 503], [501, 38], [489, 32], [11, 367]]
Point: 15 504
Point: 209 527
Point: 517 42
Point: 504 39
Point: 16 502
Saved lane_3: [[15, 504], [209, 527], [517, 42], [504, 39], [16, 502]]
Point: 212 529
Point: 412 534
Point: 542 45
Point: 517 42
Saved lane_4: [[212, 529], [412, 534], [542, 45], [517, 42]]
Saved: data/lanes.json
Point: 136 100
Point: 202 129
Point: 202 129
Point: 248 92
Point: 189 62
Point: 189 62
Point: 136 104
Saved lane_5: [[136, 100], [202, 129], [202, 129], [248, 92], [189, 62], [189, 62], [136, 104]]
Cần ít nhất 3 điểm để tạo 1 lane


: 